# Stage C — Scale-up on Colab (fast bandwidth) → features saved to Drive

Your home connection to the cancer database is ~0.3 MB/s; Colab's is far faster. This notebook
downloads the slides HERE, extracts the same features as your local pipeline, and saves the tiny
per-patient feature vectors to your **Google Drive** — so a few Colab sessions get you the
well-powered, size-matched sample your Mac couldn't.

**How to run:**
1. **Runtime > Change runtime type > CPU** (no GPU needed — avoids GPU limits).
2. **Runtime > Run all.** Approve the Google Drive popup.
3. If it disconnects, just **Run all again** — it skips patients already saved to Drive and continues.
4. Re-run across sessions until it reaches ~150–260 patients.

**Design:** size-matched case/control (all ~130 deceased + nearest-size-matched alive), so slide
size can't confound survival — identical to the corrected local analysis.

### Step 0 — Install

In [ ]:
!apt-get -qq install -y openslide-tools >/dev/null 2>&1
!pip -q install openslide-python requests >/dev/null 2>&1
import torch; print('device:', 'GPU' if torch.cuda.is_available() else 'CPU')

### Step 1 — Connect Drive (features saved here, survive disconnects)

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os
CKPT='/content/drive/MyDrive/stage_c_features'; os.makedirs(CKPT, exist_ok=True)
print('saving to', CKPT, '| already saved:', len([f for f in os.listdir(CKPT) if f.endswith('.npy')]))

### Step 2 — TCGA-BRCA survival (cBioPortal)

In [ ]:
import requests, pandas as pd
API='https://www.cbioportal.org/api'
rows=requests.get(f'{API}/studies/brca_tcga_pan_can_atlas_2018/clinical-data',
    params={'clinicalDataType':'PATIENT','projection':'SUMMARY','pageSize':10_000_000}).json()
cl=pd.DataFrame(rows).pivot_table(index='patientId',columns='clinicalAttributeId',values='value',aggfunc='first')
surv=pd.DataFrame(index=cl.index)
surv['time']=pd.to_numeric(cl['OS_MONTHS'],errors='coerce')
surv['event']=cl['OS_STATUS'].astype(str).str.startswith('1')
surv=surv.dropna(subset=['time']); surv=surv[surv['time']>0]
print('patients with survival:', len(surv))

### Step 3 — Size-matched slide list (all deaths + nearest-size alive controls)

In [ ]:
import json as _json
DEAD_N=130
filt={'op':'and','content':[
 {'op':'in','content':{'field':'cases.project.project_id','value':['TCGA-BRCA']}},
 {'op':'in','content':{'field':'data_type','value':['Slide Image']}},
 {'op':'in','content':{'field':'experimental_strategy','value':['Diagnostic Slide']}}]}
params={'filters':_json.dumps(filt),'fields':'file_id,file_size,cases.submitter_id','format':'JSON','size':'20000'}
hits=requests.get('https://api.gdc.cancer.gov/files',params=params).json()['data']['hits']
rows=[{'patient':h['cases'][0]['submitter_id'],'file_id':h['file_id'],'size':h.get('file_size',0)} for h in hits if h['cases'][0]['submitter_id'] in surv.index]
df=pd.DataFrame(rows).drop_duplicates('patient').merge(surv[['event']],left_on='patient',right_index=True)
dead=df[df.event].sort_values('size').head(DEAD_N)
pool=df[~df.event]; picks=[]; used=set()
for s in dead['size']:
    cand=pool[~pool.patient.isin(used)]
    j=(cand['size']-s).abs().idxmin(); used.add(pool.loc[j,'patient']); picks.append(j)
sl=pd.concat([dead,pool.loc[picks]]).sort_values('size').reset_index(drop=True)
print(f'selection: {len(sl)} patients | dead {int(sl.event.sum())} | alive {int((~sl.event).sum())}')
print(f'size balance: dead {sl[sl.event].size.mean()/1e6:.0f}MB vs alive {sl[~sl.event].size.mean()/1e6:.0f}MB')

### Step 4 — Feature extractor (identical to the local pipeline)

In [ ]:
import numpy as np, openslide
from torchvision.models import resnet50, ResNet50_Weights
from torchvision import transforms
DOWNSAMPLE=8; TILE=224; PATCHES=80
dev='cuda' if torch.cuda.is_available() else 'cpu'
w=ResNet50_Weights.IMAGENET1K_V2; net=resnet50(weights=w); net.fc=torch.nn.Identity(); net=net.eval().to(dev)
prep=transforms.Compose([transforms.ToTensor(), transforms.Normalize(w.transforms().mean,w.transforms().std)])
def is_tissue(im):
    a=np.asarray(im.convert('HSV')); return a[:,:,1].mean()>25 and a[:,:,2].mean()<235
def slide_vector(path):
    s=openslide.OpenSlide(path); lvl=s.get_best_level_for_downsample(DOWNSAMPLE)
    W,H=s.level_dimensions[lvl]; ds=s.level_downsamples[lvl]
    coords=[(x,y) for y in range(0,H-TILE,TILE) for x in range(0,W-TILE,TILE)]
    np.random.RandomState(0).shuffle(coords)
    tiles=[]
    for (x,y) in coords:
        reg=s.read_region((int(x*ds),int(y*ds)),lvl,(TILE,TILE)).convert('RGB')
        if is_tissue(reg): tiles.append(prep(reg))
        if len(tiles)>=PATCHES: break
    s.close()
    if not tiles: return None
    out=[]
    with torch.no_grad():
        for j in range(0,len(tiles),32): out.append(net(torch.stack(tiles[j:j+32]).to(dev)).cpu().numpy())
    return np.concatenate(out).mean(0)

### Step 5 — Download + featurize (resume-safe; re-run after any disconnect)

In [ ]:
import os
for i,r in enumerate(sl.itertuples(),1):
    fp=f'{CKPT}/{r.patient}.npy'
    if os.path.exists(fp): continue
    tmp=f'/content/{r.file_id}.svs'
    ok=False
    for attempt in range(3):
        try:
            with requests.get(f'https://api.gdc.cancer.gov/data/{r.file_id}',stream=True,timeout=(30,180)) as resp:
                resp.raise_for_status()
                with open(tmp,'wb') as fh:
                    for ch in resp.iter_content(1<<20): fh.write(ch)
            ok=True; break
        except Exception as e:
            print('  retry',attempt+1,r.patient,repr(e)[:40])
    try:
        if ok:
            v=slide_vector(tmp)
            if v is not None: np.save(fp,v)
    except Exception as e: print('  featurize fail',r.patient,repr(e)[:50])
    finally:
        if os.path.exists(tmp): os.remove(tmp)
    saved=len([f for f in os.listdir(CKPT) if f.endswith('.npy')])
    print(f'{i}/{len(sl)} | saved {saved} | {r.size/1e6:.0f}MB')
print('PASS DONE. total saved:', len([f for f in os.listdir(CKPT) if f.endswith('.npy')]))

### Step 6 — When you've got enough (~150+ patients)
The feature vectors are in your Google Drive at **MyDrive/stage_c_features/** (each ~8 KB).
Download that whole folder from drive.google.com (it's only a few MB) and hand it back — the
survival + fusion analysis runs locally in seconds. Re-run Steps 1–5 across sessions to accumulate more.